# Load Binance Recorder Data

Starter notebook for loading Binance data from the sibling `exchange-data-recorder` project.

This example is pinned to `BTCUSDC/20260222` and also shows how to preview merged market events as one DataFrame.

In [1]:
from pathlib import Path
import sys

import pandas as pd

In [2]:
def find_backtester_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "stats").is_dir() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the exchange-data-backtester project root")


PROJECT_ROOT = find_backtester_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/hoangdeveloper/PycharmProjects/exchange-data-backtester')

In [3]:
def resolve_day_dir(project_root: Path, *, symbol: str, day: str) -> Path:
    candidates = [
        project_root.parent / "exchange-data-recorder" / "data" / symbol / day,
        project_root.parent / "exchange-data-recorder" / "data" / "binance" / symbol / day,
        project_root.parent / "ExchangeDataRecorder" / "data" / symbol / day,
        project_root.parent / "ExchangeDataRecorder" / "data" / "binance" / symbol / day,
        Path("../exchange-data-recorder/data") / symbol / day,
        Path("../exchange-data-recorder/data/binance") / symbol / day,
        project_root / "data" / symbol / day,
        project_root / "data" / "binance" / symbol / day,
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "schema.json").is_file():
            return candidate
    raise FileNotFoundError(f"Could not locate a recorder day folder for {symbol}/{day}")


symbol = "BTCUSDC"
day = "20260225"
day_dir = resolve_day_dir(PROJECT_ROOT, symbol=symbol, day=day)
day_dir

PosixPath('/Users/hoangdeveloper/PycharmProjects/exchange-data-recorder/data/binance/BTCUSDC/20260225')

In [4]:
from stats.notebook import load_day_context

preview_on_gap = "skip-segment"
context = load_day_context(
    day_dir,
    include_book=False,
    include_trades=True,
    include_events=True,
    replay_on_gap=preview_on_gap,
    include_market_preview=True,
    market_preview_limit=20,
)

dataset = context["dataset"]
events = context["events"]
trades = context["trades"]
replay_info = context["replay_summary"]
market_events_preview = context["market_preview"]

dataset

In [5]:
events = dataset.load_events()
trades = dataset.load_trades()


def fmt_utc(ms: int | float) -> str:
    return pd.to_datetime(int(ms), unit="ms", utc=True).strftime("%Y-%m-%d %H:%M:%S.%f UTC")


pd.DataFrame(
    {
        "field": [
            "exchange",
            "symbol",
            "day",
            "schema_version",
            "replay_on_gap",
            "segments_total",
            "segments_kept",
            "segments_skipped",
            "events_from",
            "events_to",
            "trades_from",
            "trades_to",
            "events_path",
            "trades_path",
            "book_path",
            "diff_paths",
            "snapshot_csv_paths",
        ],
        "value": [
            dataset.exchange,
            dataset.symbol,
            dataset.day,
            dataset.metadata.schema_version,
            replay_info["replay_on_gap"],
            replay_info["segments_total"],
            replay_info["segments_kept"],
            replay_info["segments_skipped"],
            fmt_utc(events["recv_time_ms"].min()),
            fmt_utc(events["recv_time_ms"].max()),
            fmt_utc(trades["recv_time_ms"].min()),
            fmt_utc(trades["recv_time_ms"].max()),
            str(dataset.paths.events_path),
            str(dataset.paths.trades_path),
            str(dataset.paths.book_path),
            len(dataset.paths.diff_paths),
            len(dataset.paths.snapshot_csv_paths),
        ],
    }
)

,field,value
0,exchange,binance
1,symbol,BTCUSDC
2,day,20260225
3,schema_version,4
4,replay_on_gap,skip-segment
5,segments_total,2
6,segments_kept,1
7,segments_skipped,1
8,events_from,2026-02-25 01:00:04.430000 UTC
9,events_to,2026-02-25 23:15:05.049000 UTC


In [6]:
display(events.head())
display(trades.head())

,event_id,recv_time_ms,recv_seq,run_id,type,epoch_id,details_json
0,1771981204431,1771981204430,1,1771981204424,run_start,0,"{""symbol"": ""BTCUSDC"", ""symbol_fs"": ""BTCUSDC"", ..."
1,1771981204432,1771981204431,2,1771981204424,ws_connecting,0,"{""attempt"": 1}"
2,1771981204433,1771981204476,3,1771981204424,state_change,0,"{""from"": ""connecting"", ""to"": ""snapshot"", ""reas..."
3,1771981204434,1771981204476,4,1771981204424,ws_open,0,"{""ws_url"": ""wss://stream.binance.com:9443/stre..."
4,1771981204435,1771981204477,5,1771981204424,snapshot_request,0,"{""tag"": ""initial"", ""limit"": 1000}"


,event_time_ms,recv_time_ms,recv_seq,run_id,trade_id,trade_time_ms,price,qty,is_buyer_maker,side,ord_type,exchange,symbol
0,1771981205233,1771981205236,24,1771981204424,336863843,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC
1,1771981205233,1771981205237,25,1771981204424,336863844,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC
2,1771981205233,1771981205237,26,1771981204424,336863845,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC
3,1771981205233,1771981205237,27,1771981204424,336863846,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC
4,1771981205233,1771981205237,28,1771981204424,336863847,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC


## Cached Derived Tables

These helpers build Parquet-backed analysis tables under `DAY_DIR/cache/v2/`. On rerun, they load cached tables instead of replaying the day from raw inputs again.

In [7]:
from stats.tables import (
    get_or_build_market_grid,
    get_or_build_top_of_book_table,
    get_or_build_trades_table,
)

top_of_book = get_or_build_top_of_book_table(dataset, on_gap=preview_on_gap)
trades_enriched = get_or_build_trades_table(dataset)
market_grid = get_or_build_market_grid(dataset, grid_freq="100ms", on_gap=preview_on_gap)

display(top_of_book.head())
display(trades_enriched.head())
display(market_grid.head())

,event_type,recv_seq,recv_time_ms,event_time_ms,epoch_id,segment_index,segment_tag,bid1_price,bid1_qty,ask1_price,ask1_qty,mid,spread,spread_bps,microprice,ts
0,book,17,1771981204734,1771981204732,1,1,resync_000001,64243.01,0.18858,64243.02,0.54737,64243.015,0.01,0.001557,64243.012562,2026-02-25 01:00:04.734000+00:00
1,book,19,1771981204834,1771981204832,1,1,resync_000001,64243.01,0.18858,64243.02,0.54737,64243.015,0.01,0.001557,64243.012562,2026-02-25 01:00:04.834000+00:00
2,book,20,1771981204935,1771981204932,1,1,resync_000001,64243.01,0.18858,64243.02,0.63977,64243.015,0.01,0.001557,64243.012277,2026-02-25 01:00:04.935000+00:00
3,book,21,1771981205035,1771981205032,1,1,resync_000001,64243.01,0.22762,64243.02,0.03586,64243.015,0.01,0.001557,64243.018639,2026-02-25 01:00:05.035000+00:00
4,book,22,1771981205135,1771981205132,1,1,resync_000001,64243.01,0.26308,64243.02,0.09820,64243.015,0.01,0.001557,64243.017282,2026-02-25 01:00:05.135000+00:00


,event_time_ms,recv_time_ms,recv_seq,run_id,trade_id,trade_time_ms,price,qty,is_buyer_maker,side,ord_type,exchange,symbol,aggr_sign,signed_qty,notional,signed_notional,ts,trade_ts
0,1771981205233,1771981205236,24,1771981204424,336863843,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC,1.0,0.00008,5.139442,5.139442,2026-02-25 01:00:05.236000+00:00,2026-02-25 01:00:05.233000+00:00
1,1771981205233,1771981205237,25,1771981204424,336863844,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC,1.0,0.00008,5.139442,5.139442,2026-02-25 01:00:05.237000+00:00,2026-02-25 01:00:05.233000+00:00
2,1771981205233,1771981205237,26,1771981204424,336863845,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC,1.0,0.00008,5.139442,5.139442,2026-02-25 01:00:05.237000+00:00,2026-02-25 01:00:05.233000+00:00
3,1771981205233,1771981205237,27,1771981204424,336863846,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC,1.0,0.00008,5.139442,5.139442,2026-02-25 01:00:05.237000+00:00,2026-02-25 01:00:05.233000+00:00
4,1771981205233,1771981205237,28,1771981204424,336863847,1771981205233,64243.02,0.00008,0,buy,NaN,binance,BTCUSDC,1.0,0.00008,5.139442,5.139442,2026-02-25 01:00:05.237000+00:00,2026-02-25 01:00:05.233000+00:00


,event_type,recv_seq,recv_time_ms,event_time_ms,epoch_id,segment_index,segment_tag,bid1_price,bid1_qty,ask1_price,...,microprice,book_updates,valid_book,trade_count,total_qty,signed_qty,total_notional,signed_notional,tfi_qty,tfi_notional
ts,,,,,,,,,,,,,,,,,,,,,
2026-02-25 01:00:04.700000+00:00,book,17.0,1.771981e+12,1.771981e+12,1.0,1.0,resync_000001,64243.01,0.18858,64243.02,...,64243.012562,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-25 01:00:04.800000+00:00,book,19.0,1.771981e+12,1.771981e+12,1.0,1.0,resync_000001,64243.01,0.18858,64243.02,...,64243.012562,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-25 01:00:04.900000+00:00,book,20.0,1.771981e+12,1.771981e+12,1.0,1.0,resync_000001,64243.01,0.18858,64243.02,...,64243.012277,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-25 01:00:05+00:00,book,21.0,1.771981e+12,1.771981e+12,1.0,1.0,resync_000001,64243.01,0.22762,64243.02,...,64243.018639,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-25 01:00:05.100000+00:00,book,22.0,1.771981e+12,1.771981e+12,1.0,1.0,resync_000001,64243.01,0.26308,64243.02,...,64243.017282,1,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Merged Market Events Preview

The core backtest API is an iterator, but for inspection it is useful to convert a slice of it into one DataFrame.

Some days contain an invalid initial segment followed by recorder-driven resync. For exploratory work, use `on_gap="skip-segment"` so the preview starts from the first valid replay segment.

In [8]:
market_events_preview

,event_type,recv_seq,recv_time_ms,event_time_ms,epoch_id,segment_index,segment_tag,bid1_price,bid1_qty,ask1_price,...,trade_time_ms,trade_id,run_id,price,qty,side,is_buyer_maker,exchange,symbol,ord_type
0,book,17,1771981204734,1771981204732,1,1,resync_000001,64243.01,0.18858,64243.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,book,19,1771981204834,1771981204832,1,1,resync_000001,64243.01,0.18858,64243.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,book,20,1771981204935,1771981204932,1,1,resync_000001,64243.01,0.18858,64243.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,book,21,1771981205035,1771981205032,1,1,resync_000001,64243.01,0.22762,64243.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,book,22,1771981205135,1771981205132,1,1,resync_000001,64243.01,0.26308,64243.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,book,23,1771981205234,1771981205232,1,1,resync_000001,64243.01,0.28308,64243.02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,trade,24,1771981205236,1771981205233,1,1,resync_000001,NaN,NaN,NaN,...,1.771981e+12,336863843.0,1.771981e+12,64243.02,0.00008,buy,0.0,binance,BTCUSDC,NaN
7,trade,25,1771981205237,1771981205233,1,1,resync_000001,NaN,NaN,NaN,...,1.771981e+12,336863844.0,1.771981e+12,64243.02,0.00008,buy,0.0,binance,BTCUSDC,NaN
8,trade,26,1771981205237,1771981205233,1,1,resync_000001,NaN,NaN,NaN,...,1.771981e+12,336863845.0,1.771981e+12,64243.02,0.00008,buy,0.0,binance,BTCUSDC,NaN
9,trade,27,1771981205237,1771981205233,1,1,resync_000001,NaN,NaN,NaN,...,1.771981e+12,336863846.0,1.771981e+12,64243.02,0.00008,buy,0.0,binance,BTCUSDC,NaN


In [9]:
# Optional: materialize the full merged stream.
# This can be large for active days.
#
# from stats.notebook import load_market_preview
# market_events_df = load_market_preview(dataset, limit=1000000, replay_on_gap=preview_on_gap)
# market_events_df.head()